In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [2]:
hormoneReport = pd.read_csv(r"C:\Users\direk\OneDrive\Desktop\data_diseasePredictor\hormone_report.csv")
hormoneReport = hormoneReport.drop("risk_label", axis=1)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol
0,2.86,113.68,6.02,464.25,101.60,-3.04,7.36,4.82,3.07,13.67
1,2.33,182.48,7.68,100.90,-28.46,22.84,10.95,8.22,6.90,16.97
2,5.87,97.79,3.77,489.16,74.57,4.38,9.59,4.37,2.79,14.03
3,3.57,88.82,3.80,317.35,112.12,-15.80,4.66,4.50,3.20,20.96
4,2.32,151.14,5.61,321.28,35.22,15.04,6.27,3.21,3.04,7.65


In [3]:
def thyroid_disorder(row):
    tsh, t3, t4 = row["tsh"], row["t3"], row["t4"]

    if tsh > 10 or t3 < 50 or t4 < 3:
        return "Critical"
    elif tsh > 6 or t3 < 70 or t4 < 4:
        return "High"
    elif tsh > 4 or t3 < 80 or t4 < 4.5:
        return "Medium"
    else:
        return "Low"


def testosterone_imbalance(v):
    if v < 150:
        return "Critical"
    elif v < 250:
        return "High"
    elif v < 300 or v > 1000:
        return "Medium"
    else:
        return "Low"


def hormonal_imbalance(row):
    score = 0

    if (row[["estradiol", "progesterone", "prolactin", "lh", "fsh", "cortisol"]] < -10).any():
        return "Critical"

    if (row[["estradiol", "progesterone", "prolactin", "lh", "fsh", "cortisol"]] < 0).any():
        return "High"

    if row["prolactin"] > 25:
        score += 1
    if row["cortisol"] > 30:
        score += 1
    if row["estradiol"] > 200:
        score += 1

    ratio = row["lh"] / (row["fsh"] + 1e-5)
    if ratio > 3:
        score += 1

    if score >= 2:
        return "High"
    elif score == 1:
        return "Medium"
    else:
        return "Low"

In [4]:
hormoneReport["thyroid_disorder"] = hormoneReport.apply(thyroid_disorder, axis=1)
hormoneReport["testosterone_imbalance"] = hormoneReport["testosterone"].apply(testosterone_imbalance)
hormoneReport["hormonal_imbalance"] = hormoneReport.apply(hormonal_imbalance, axis=1)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol,thyroid_disorder,testosterone_imbalance,hormonal_imbalance
0,2.86,113.68,6.02,464.25,101.60,-3.04,7.36,4.82,3.07,13.67,Low,Low,High
1,2.33,182.48,7.68,100.90,-28.46,22.84,10.95,8.22,6.90,16.97,Low,Critical,Critical
2,5.87,97.79,3.77,489.16,74.57,4.38,9.59,4.37,2.79,14.03,High,Low,Low
3,3.57,88.82,3.80,317.35,112.12,-15.80,4.66,4.50,3.20,20.96,High,Low,Critical
4,2.32,151.14,5.61,321.28,35.22,15.04,6.27,3.21,3.04,7.65,Low,Low,Low


In [5]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [6]:
hormoneReport["thyroid_disorder"] = hormoneReport["thyroid_disorder"].map(LABEL_MAPPING)
hormoneReport["testosterone_imbalance"] = hormoneReport["testosterone_imbalance"].map(LABEL_MAPPING)
hormoneReport["hormonal_imbalance"] = hormoneReport["hormonal_imbalance"].map(LABEL_MAPPING)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol,thyroid_disorder,testosterone_imbalance,hormonal_imbalance
0,2.86,113.68,6.02,464.25,101.60,-3.04,7.36,4.82,3.07,13.67,low,low,high
1,2.33,182.48,7.68,100.90,-28.46,22.84,10.95,8.22,6.90,16.97,low,critical,critical
2,5.87,97.79,3.77,489.16,74.57,4.38,9.59,4.37,2.79,14.03,high,low,low
3,3.57,88.82,3.80,317.35,112.12,-15.80,4.66,4.50,3.20,20.96,high,low,critical
4,2.32,151.14,5.61,321.28,35.22,15.04,6.27,3.21,3.04,7.65,low,low,low


In [7]:
hormoneReport["thyroid_disorder_num"] = hormoneReport["thyroid_disorder"].map(NUM_MAPPING)
hormoneReport["testosterone_imbalance_num"] = hormoneReport["testosterone_imbalance"].map(NUM_MAPPING)
hormoneReport["hormonal_imbalance_num"] = hormoneReport["hormonal_imbalance"].map(NUM_MAPPING)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol,thyroid_disorder,testosterone_imbalance,hormonal_imbalance,thyroid_disorder_num,testosterone_imbalance_num,hormonal_imbalance_num
0,2.86,113.68,6.02,464.25,101.60,-3.04,7.36,4.82,3.07,13.67,low,low,high,0,0,2
1,2.33,182.48,7.68,100.90,-28.46,22.84,10.95,8.22,6.90,16.97,low,critical,critical,0,3,3
2,5.87,97.79,3.77,489.16,74.57,4.38,9.59,4.37,2.79,14.03,high,low,low,2,0,0
3,3.57,88.82,3.80,317.35,112.12,-15.80,4.66,4.50,3.20,20.96,high,low,critical,2,0,3
4,2.32,151.14,5.61,321.28,35.22,15.04,6.27,3.21,3.04,7.65,low,low,low,0,0,0


In [8]:
"""Preparring data for ML prediction"""
feature_cols = ["tsh","t3", "t4", "testosterone", "estradiol", "progesterone", "prolactin", "lh", "fsh", "cortisol"]
X = hormoneReport[feature_cols]
y_thy = hormoneReport["thyroid_disorder_num"]
y_tes = hormoneReport["testosterone_imbalance_num"]
y_hor = hormoneReport["hormonal_imbalance_num"]

In [9]:
"""train test split"""
X_train, X_test, y_train_thy, y_test_thy = train_test_split(X, y_thy, test_size=0.2, random_state=31, stratify=y_thy)
X_train, X_test, y_train_tes, y_test_tes = train_test_split(X, y_tes, test_size=0.2, random_state=31, stratify=y_tes)
X_train, X_test, y_train_hor, y_test_hor = train_test_split(X, y_hor, test_size=0.2, random_state=31, stratify=y_hor)


In [10]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))

In [11]:
"""Thyroid disorder model"""
model_thyroid = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_thyroid.fit(X_train, y_train_thy)
y_predict_throid = model_thyroid.predict(X_test)
print_report(y_test_thy, y_predict_throid, "THROID DISORDER")

THROID DISORDER MODEL ACCURACY
Accuracy: 55.00%

THROID DISORDER CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.60      0.89      0.72        61
    moderate       0.33      0.06      0.10        18
        high       0.00      0.00      0.00        16
    critical       0.00      0.00      0.00         5

    accuracy                           0.55       100
   macro avg       0.23      0.24      0.20       100
weighted avg       0.43      0.55      0.45       100



In [12]:
"""Testosterone imbalance model"""
model_testosterone = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_testosterone.fit(X_train, y_train_tes)
y_predict_testosterone = model_testosterone.predict(X_test)
print_report(y_test_tes, y_predict_testosterone, "TESTOSTERONE IMBALANCE")

TESTOSTERONE IMBALANCE MODEL ACCURACY
Accuracy: 72.00%

TESTOSTERONE IMBALANCE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.73      0.97      0.84        74
    moderate       0.00      0.00      0.00        10
        high       0.00      0.00      0.00        11
    critical       0.00      0.00      0.00         5

    accuracy                           0.72       100
   macro avg       0.18      0.24      0.21       100
weighted avg       0.54      0.72      0.62       100



In [13]:
"""Hormonal imbalance model"""
model_hormone = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)

model_hormone.fit(X_train, y_train_hor)
y_predict_hormone = model_hormone.predict(X_test)
print_report(y_test_hor, y_predict_hormone, "HORMONAL IMBALANCE")

HORMONAL IMBALANCE MODEL ACCURACY
Accuracy: 96.00%

HORMONAL IMBALANCE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.98      0.98      0.98        52
    moderate       0.82      0.82      0.82        11
        high       0.96      0.96      0.96        25
    critical       1.00      1.00      1.00        12

    accuracy                           0.96       100
   macro avg       0.94      0.94      0.94       100
weighted avg       0.96      0.96      0.96       100



In [14]:
"""Save both models as pkl"""
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, "hormone.pkl"), "wb") as f:
    pickle.dump(model_hormone, f)
    print("hormone.pkl saved successfully!")

with open(os.path.join(save_path, "testosterone.pkl"), "wb") as f:
    pickle.dump(model_testosterone, f)
    print("testosterone.pkl saved sucessfully!")

with open(os.path.join(save_path, "thyroid.pkl"), "wb") as f:
    pickle.dump(model_thyroid, f)
    print("thyroid.pkl saved sucessfully!")


hormone.pkl saved successfully!
testosterone.pkl saved sucessfully!
thyroid.pkl saved sucessfully!
